In [1]:
import sys
from typing import List
import requests
import re
import numpy as np
from alfworld.agents.environment import get_environment
import alfworld.agents.modules.generic as generic


sys.argv = ['jupyter_notebook.py', 'configs/eval_config.yaml']
config = generic.load_config()
# env_type = config['env']['type']
# env = get_environment(env_type)(config, train_eval='train')
# env = env.init_env(batch_size=1)

In [2]:
from alfworld.agents.environment import get_environment
from alfworld.agents.environment.alfred_tw_env import TASK_TYPES
num_games = 50
eval_split = 'eval_out_of_distribution'
def load_games(game_files: List[str], num_games: int) -> List[str]:
    """Select games with uniform coverage across task types.

    If num_games <= 0, return all games sorted.
    Otherwise, take floor(num_games/num_types) per type,
    distribute the remainder round-robin by type alphabetical order.
    Within each type, take the first N (sorted) games.
    """
    files = sorted(game_files)
    if num_games <= 0 or num_games >= len(files):
        return files

    # Group by task type
    groups = {}
    for f in files:
        tt = detect_task_type(f)
        groups.setdefault(tt, []).append(f)

    types = sorted(groups.keys())
    per_type = num_games // len(types)
    remainder = num_games % len(types)

    selected = []
    for i, tt in enumerate(types):
        take = per_type + (1 if i < remainder else 0)
        selected.extend(groups[tt][:take])

    return sorted(selected)


def detect_task_type(gamefile_path: str) -> str:
    """Detect task type from the game file path."""
    for task_type in TASK_TYPES.values():
        if task_type in gamefile_path:
            return task_type
    return "unknown"

env_type = config["env"]["type"]
AlfredEnvClass = get_environment(env_type)
alfred_env = AlfredEnvClass(config, train_eval=eval_split)
alfred_env.game_files = load_games(alfred_env.game_files, num_games)
alfred_env.num_games = len(alfred_env.game_files)

# env = alfred_env.init_env(batch_size=1)

Initializing AlfredTWEnv...


100%|██████████| 341/341 [00:00<00:00, 1316.84it/s]

Overall we have 134 games in split=eval_out_of_distribution
Evaluating with 134 games


In [3]:
def call_llm(prompt):
    url = "https://tritonai-api.ucsd.edu/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": "Bearer " + "sk-Z54pOkD5b_Y6b5jVORpDuw"
    }

    payload = {
    "model": "api-llama-4-scout",
    "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        result = response.json()
        return result['choices'][0]['message']['content'].strip()
    else:
        return f"API Error {response.status_code}: {response.text}"

## BaseLine 

In [4]:

    
def alf_agent(task, obs, commands):
    prompt = f"""You are a agent for a Text-based household simulation game
    Task: {task}
    Observation: {obs}
    valid command: {commands}
    You MUST return the text of the command you choose.
    """
    return call_llm(prompt)


### Evaluation

For baseline, or only LLM agent, the success rate is most likely 0.

In [5]:
# success_count = 0
# num_test_games = alfred_env.num_games
# MAX_STEPS = 50
# all_results = []

# for game_idx in range(num_test_games):
#     obs, info = env.reset()
#     task = obs[0].split("Your task is to: ")[1].strip()
#     valid_commands = list(info['admissible_commands'][0])

#     for _ in range(MAX_STEPS):
#         response = alf_agent(task, obs[0], valid_commands)
#         action = response
#         for cmd in valid_commands:
#             if cmd in response.lower():
#                 action = cmd
#                 break
#         obs, scores, dones, infos = env.step([action])
#         info = infos
#         if dones[0]:
#             if scores[0] > 0:
#                 success_count += 1
#                 print(f"success")
#             else:
#                 print(f"failure")
#             break
#         all_results.append(scores[0] > 0)

# final_sr = (success_count / num_test_games) * 100
# print(final_sr)

## LLM + Memory

In [6]:
def alf_agent_short_mem(task, obs, commands, history):
    recent_history = "\n".join([f"- {m}" for m in history]) if history else "None"
    
    prompt = f"""You are a agent for a Text-based household simulation game
    Task: {task}
    Observation: {obs}
    valid command: {commands}
    recent history: {recent_history}

    INSTRUCTION:
    1. Try to first find the object the task need.
    2. Check your history: Have you already tried this action? If it said "Nothing happens", do NOT repeat it.
    3. If you don't have the object, you must EXPLORE other locations.
    4. You MUST return the text of the command you choose.
    Output Format:
    Thought: <one sentence reasoning>
    Action: <command>
    """
    return call_llm(prompt)

### short_term history Evaluation

In [7]:
# success_count = 0
# num_test_games = alfred_env.num_games
# MAX_STEPS = 50
# MEMORY_SIZE = 3
# all_results = []

# for game_idx in range(num_test_games):
#     obs, info = env.reset()
#     task = obs[0].split("Your task is to: ")[1].strip()
#     valid_commands = list(info['admissible_commands'][0])
#     short_memory = []

#     for _ in range(MAX_STEPS):
#         response = alf_agent_short_mem(task, obs[0], valid_commands, short_memory)
#         action = response
#         for cmd in valid_commands:
#             if cmd in response.lower():
#                 action = cmd
#                 break

#         obs, scores, dones, infos = env.step([action])
#         current_step_record = f"Action: {action} -> Result: {obs[0]}"
#         short_memory.append(current_step_record)
#         if len(short_memory) > MEMORY_SIZE:
#             short_memory.pop(0)

#         info = infos
#         if dones[0]:
#             if scores[0] > 0:
#                 success_count += 1
#                 print(f"success")
#             else:
#                 print(f"failure")
#             break
#         all_results.append(scores[0] > 0)

# final_sr = (success_count / num_test_games) * 100
# print(final_sr)
# 1. If you need an object and its location is in 'Found Objects', GO THERE.
#         2. If the object's location is unknown, go to an unexplored location that is likely to contain it.
#         3. If you are looking at a location and the object is NOT in the Observation, IT IS NOT THERE. You must move to another location.
#         4. If your Recent history shows an action resulted in "Nothing happens" or a failed state, DO NOT repeat it.

## Memory + stateful knowledge base


In [8]:
def alf_agent_short_mem_kb(task, obs, commands, history, kb):
    unvisited = [c for c in commands if "go to" in c and c.split("go to ")[1] not in kb['visited_locations']]

    kb_summary = f"""
        - Current Inventory: {kb['inventory']}
        - Visited Locations: {list(kb['visited_locations'])}
        - Found Objects (Items you saw but didn't take): {kb['seen_items']}
        - Unexplored (top 8): {unvisited[:8]}
    """

    recent_history = "\n".join([f"- {m}" for m in history]) if history else "None"
    
    prompt = f"""You are a agent for a Text-based household simulation game
    Task: {task}
    Observation: {obs}
    Valid commands: {commands}
    Recent history: {recent_history}
    Knowledge base: {kb_summary}

    INSTRUCTION:
    1. RECOGNITION: List exactly what you see in the current Observation. If the target object is NOT in that list, you CANNOT take it.
    2. VALIDATION: Look at 'VALID COMMANDS'. You can ONLY pick an action from this list.
    3. HISTORY CHECK: If your Recent history shows an action resulted in "Nothing happens" or a failed state, DO NOT repeat it.
    4. EXPLORATION: If you don't have the item and it's not in the Observation, use a 'go to' command ONLY from 'Unexplored' to find it. 
    
    Output ONLY one command from the valid commands list.
    Do not include explanations.
    
    Format:
    Thought: <Briefly explain: 1. Do I see the item? 2. What did I try that failed? 3. Where should I go next?>
    Action: <command>
    """
    return call_llm(prompt)

In [9]:
def update_kb(action, observation, kb):
    obs_text = observation[0] if isinstance(observation, (list, tuple)) else observation

    arrival_match = re.search(r"You arrive at ([\w\s\d\-]+)\.", obs_text)
    if arrival_match:
        loc = arrival_match.group(1).strip()
        kb["current_location"] = loc
        kb["visited_locations"].add(loc)
    
    if "you see" in obs_text:
        items_part = obs_text.split("you see")[-1].strip().rstrip(".")
        items = [i.strip() for i in re.split(r', and a|, a|and a|a |an ', items_part) if i.strip()]
        if kb["current_location"] != "unknown":
            kb["seen_items"].setdefault(kb["current_location"], []).extend(items)

    if "You pick up" in obs_text:
        item_match = re.search(r"You pick up the ([\w\s\d]+) from", obs_text)
        if item_match:
            item = item_match.group(1).strip()
            kb["inventory"].add(item)
            loc = kb["current_location"]
            if loc in kb["seen_items"] and item in kb["seen_items"][loc]:
                kb["seen_items"][loc].remove(item)
    

    m = re.match(r"^(put|move)\s+(.+?)\s+to\s+(.+)$", action.lower())
    if m and ("You put" in obs_text or "You move" in obs_text):
        obj = m.group(2).strip()
        kb["inventory"] = {x for x in kb["inventory"] if x.lower() != obj}
        
    if kb["current_location"] == "unknown":
        print("Warning: Current location is unknown. Observation may be incomplete.")
    return kb

def extract_action_from_response(response):
    response = response.strip()
    if "Action:" in response:
        action_text = response.split("Action:")[-1].strip()
    else:
        action_text = response.strip()
    
    action_text = action_text.rstrip('.,!?;:\n').strip().lower()
    return action_text

def validate_and_match_action(action_text, valid_commands):
    action_text = action_text.lower().strip()
    for cmd in valid_commands:
        if cmd.lower() == action_text:
            return cmd
        
    for cmd in valid_commands:
        if action_text in cmd.lower():
            return cmd
        
    return None

In [10]:
import os
from datetime import datetime
import json
log_dir = "eval_logs"
os.makedirs(log_dir, exist_ok=True)
log_file = os.path.join(log_dir, f"eval_results_{datetime.now().strftime('%m%d_%H%M')}.json")

In [11]:
success_count = 0
num_test_games = alfred_env.num_games
MAX_STEPS = 50
MEMORY_SIZE = 3
all_results = []
tes = 25
all_game_logs = []


env = alfred_env.init_env(batch_size=1)
for game_idx in range(num_test_games):
    obs, info = env.reset()
    task = obs[0].split("Your task is to: ")[1].strip()
    
    short_memory = []
    kb = {
        "inventory": set(),       
        "visited_locations": set(),  
        "seen_items": {},      
        "current_location": "unknown" 
    }
    current_game_record = {
        "game_idx": game_idx,
        "task": task,
        "steps": [],
        "success": False
    }
    for step_num in range(MAX_STEPS):
        valid_commands = list(info['admissible_commands'][0])
        response = alf_agent_short_mem_kb(task, obs[0], valid_commands, short_memory, kb)
        action_text = extract_action_from_response(response)
        action = validate_and_match_action(action_text, valid_commands)

        if action == None:
            action = "look" 
            current_step_record = f"Action: {action_text} -> Result: ERROR: This command is not in the Valid Commands list. Pick something else."

        obs, scores, dones, infos = env.step([action])
        current_step_record = f"Action: {action} -> Obs: {obs[0][:100]}"
        short_memory.append(current_step_record)
        if len(short_memory) > MEMORY_SIZE:
            short_memory.pop(0)

        info = infos
        kb = update_kb(action, obs[0], kb)

        step_data = {
            "step": step_num + 1,
            "llm_thought": response.split("Action:")[0].replace("Thought:", "").strip(),
            "chosen_action": action,
            "observation": obs,
            "score": scores[0]
        }
        current_game_record["steps"].append(step_data)
        if dones[0]:
            if scores[0] > 0:
                success_count += 1
                print(f"success")
            else:
                print(f"failure")
            break
    all_game_logs.append(current_game_record)
    with open(log_file, 'w', encoding='utf-8') as f:
        json.dump(all_game_logs, f, indent=4, ensure_ascii=False)

final_sr = (success_count / num_test_games) * 100
print(final_sr)

failure
failure


KeyboardInterrupt: 

In [ ]:
# sys.argv = ['jupyter_notebook.py', 'configs/eval_config.yaml']
# config = generic.load_config()
# env_type = config['env']['type']
# env = get_environment(env_type)(config, train_eval='train')
# env = env.init_env(batch_size=1)

# obs, info = env.reset()
# MAX_STEPS = 10
# MEMORY_SIZE = 3
# task = obs[0].split("Your task is to: ")[1].strip()
# short_memory = []

# kb = {
#     "inventory":  set(),       
#     "visited_locations": set(),  
#     "seen_items": {},      
#     "current_location": "unknown" 
# }
# print(f"Start agent, task{task}")

# for i in range(MAX_STEPS):
#     valid_commands = list(info['admissible_commands'][0])

#     response = alf_agent_short_mem_kb(task, obs[0], valid_commands, short_memory, kb)
#     action = response
#     if "Action:" in response:
#         action_text = response.split("Action:")[-1].strip().lower()
#     else:
#         action_text = response.strip().lower()
    
#     action = None
#     for cmd in valid_commands:
#         if cmd.lower() == action_text:
#             action = cmd
#             break
#     if action == None:
#         action = np.random.choice(valid_commands)

#     obs, scores, dones, infos = env.step([action])
#     current_step_record = f"Action: {action} -> Result: {obs[0]}"
#     short_memory.append(current_step_record)
#     if len(short_memory) > MEMORY_SIZE:
#         short_memory.pop(0)
#     print("-"*80)
#     print(f"{i+1} step")
#     print(f"LLM thinking: {response}")
#     print(f"chosen Action: {action}")
#     print(f"Obs: {obs}, Score: {scores[0]}")
#     print("-"*80)
#     info = infos
#     kb = update_kb(action, obs[0], kb)
#     if dones[0]:
#         print(f"\nGame Over! The LLM finished with a score of: {scores[0]}")
#         break
